# Analysis of Household Energy Consumption in Germany
**Author:** Maximilian Starp  
**Dataset Source:** [Statistisches Bundesamt (Destatis) / Link here]

### Executive Summary
This project analyzes the shifts in energy consumption forms within German households over the last decades. 
The focus lies on the transition from fossil fuels towards renewable energy sources.

### Research Questions
Is the transition from fossil to renewable energy impacting socio-economic fairness? Specifically, is the burden of combating climate change being disproportionately carried by lower-income populations?

In [41]:
# import requirements

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Step 1: Data Acquisition & Loading

In this section, the six core datasets required for the socio-economic impact analysis are imported:

- Household Energy Consumption by Source: Detailed breakdown of energy forms (Electricity, Gas, Renewables, etc.) consumed by German households. The Data has been measured yearly

- Equivalence Income Distribution: Data categorized by income deciles/Quantiles/Quartiles etc. The Data has been measure yearly
    - Definition: The Equivalised Disposable Income is the total income of a household, after tax and other deductions, divided by the number of household members converted into equalised adults (using the OECD scale).

- Electricity Pricing (Historical & Current): Two-part dataset covering price evolution from 1991–2007 and 2007–2025, the data is given for Germany and EU. The Data has been measeured half-yearly.

- Natural Gas Pricing (Heating): Two-part dataset reflecting consumer costs for residential heating from 1985–2007 and 2007–2025, the data is given for Germany and EU. The Data has been measured half-yearly

In [42]:
# --- Load the Datasets ---

data_files = {
    "energy_mix": "EnergyMix_Households_1990-2024_Germany_annual_KWH.csv",
    "income": "EquivalisedIncome_PerCapita_1995-2025_Germany_EU_annual_EUR.csv",
    "elec_prices_old": "ElectricityPrices_Households_1991-2007_Germany-EU_biannual_KWH.csv",
    "elec_prices_new": "ElectricityPrices_Households_2007-2025_Germany-EU_biannual_KWH.csv",
    "gas_prices_old": "GasPrices_Households_1985-2007_Germany-EU_biannual_GJ.csv",
    "gas_prices_new": "GasPrices_Households_2007-2025_Germany-EU_biannual_KWH.csv",
}

datasets = {}

for key, file in data_files.items():
    try:
        datasets[key] = pd.read_csv(f"./data/{file}")
        print(f"Successfully loaded {key}")
    except FileNotFoundError:
        print(f"Error: {file} not found in directory.")

Successfully loaded energy_mix
Successfully loaded income
Successfully loaded elec_prices_old
Successfully loaded elec_prices_new
Successfully loaded gas_prices_old
Successfully loaded gas_prices_new


# Step 2: Data Harmonization Pipeline - Germany & EU (1995–2024)

This script processes four distinct types of data: **Energy Mix (Consumption of Households)**, **Electricity Prices (for Households)**, **Gas Prices (for Households)**, and **Equivalised Income (per Capita)**. It performs unit normalization, handles methodological changes from the year 2007, and transforms the data from **Long** to **Wide** format for easier analysis.

---

## 1. Global Parameters

* **Timeframe**: The data is strictly filtered for **1995 to 2024**, because in this the biggest Timeframe all Datasets are available

* **Primary Helper (`get_clean_year`)**: Eurostat often uses strings like `"2007-S1"` (Semester 1). This function extracts the first 4 characters and converts them to integers to allow for numerical filtering, also if something is measure biannualy the mean is taken form the both semester to get a annual value.

---

## 2. Energy Mix Processing

**Goal**: Convert household energy consumption into a consistent unit (**kWh**).

* **Unit Conversion**: The raw data is provided in **Terawatt-hours (TWh)**. Since prices are measured in **Kilowatt-hours (kWh)**, the script multiplies all energy values by **1,000,000,000 (10^9)**.

---

## 3. Electricity & Gas Price Harmonization (`process_prices`)

Processing energy prices is complex because Eurostat changed its reporting methodology in **2007**.

### Methodology Stitching (2007 Logic)

* Data **before 2007** is taken from the **Old** dataset.
* Data **from 2007 onwards** is taken from the **New** dataset.
* This prevents overlaps and ensures the most modern data is used for the transition year.

### Gas Unit Normalization

* Old gas data is often in **Gigajoules (GJ)**.
* To match the modern **kWh** standard, the script divides the old values by **277.78**.

### Consumption Band Filtering

To avoid duplicates, the script filters for standard household consumption bands:

* **Electricity**: Band DC (2,500–5,000 kWh/year)
* **Gas**: Band D2 / D3

### Tax Differentiation

* Columns are split to distinguish between:

  * Prices **including all taxes**
  * Prices **excluding taxes**

### Temporal Aggregation

* Price data is **bi-annual** (S1 and S2).
* The script calculates the **annual mean** for each year.

### Dual-Region Filtering
The script identifies and separates data for **Germany** and the **European Union (EU)**.  
This enables direct benchmarking of German energy trends against the European average.

### Column Structure
This logic transforms the vertical Eurostat data into clear, side-by-side columns, for example:

- `Price_Germany_WithTaxes_perKWh`
- `Price_Germany_WithoutTaxes_perKWh`
- `Price_EU_WithTaxes_perKWh`
- `Price_EU_WithoutTaxes_perKWh`

This structure makes it easy to calculate the **tax burden percentage** for any given year.
---

## 4. Income Data Transformation

**Goal**: Convert the vertical Eurostat list into a horizontal table of income deciles.

### Pivot Table

The script transforms the quantile column. Instead of multiple rows per year, it creates a single row with multiple columns:

* `Income_Germany_First_decile`
* `Income_Germany_Second_decile`
* ...
* `Income_EU_First_decile`
* ...

### Metric

* Uses the **Top cut-off point** (or mean where applicable) to represent income levels of different socio-economic groups.

---

## 5. Final Output Structure

The result of this pipeline is four clean DataFrames ready for correlation analysis:

* `df_energy_final`: Annual consumption per carrier in **kWh**
* `df_elec_final`: Annual mean electricity prices in **EUR/kWh**
* `df_gas_final`: Annual mean gas prices in **EUR/kWh**
* `df_income_final`: Annual income thresholds for all deciles in **EUR**


In [48]:
START_YEAR = 1995
END_YEAR = 2024

# --- 1. Helper Functions ---
def get_clean_year(df):
    if 'TIME_PERIOD' in df.columns:
        return df['TIME_PERIOD'].astype(str).str[:4].astype(int)
    return df['Year']

def process_prices(df_old_raw, df_new_raw, is_gas=False):
    df_old = df_old_raw.copy()
    df_new = df_new_raw.copy()
    
    # Standardize Years
    df_old['Year'] = get_clean_year(df_old)
    df_new['Year'] = get_clean_year(df_new)
    
    # IMPORTANT: Filter for the standard household consumption band to avoid duplicates
    # Electricity: Band DC (2500-5000 kWh) | Gas: Band D2 (20-200 GJ)
    if is_gas:
        # Filter for the standard band in both old and new formats
        df_old = df_old[df_old['consom'].str.contains('D3', na=False)] 
        df_new = df_new[df_new['nrg_cons'].str.contains('D2', na=False)]
    else:
        df_old = df_old[df_old['consom'].str.contains('Dc', na=False)]
        df_new = df_new[df_new['nrg_cons'].str.contains('DC', na=False)]

    # 2007 Priority Logic: Remove 2007 from old, keep in new
    df_old = df_old[df_old['Year'] < 2007]
    
    # Reset index before concatenation to avoid "duplicate labels" error
    combined = pd.concat([df_old, df_new], ignore_index=True, sort=False)
    
    # Unit conversion for Gas (GJ -> KWh)
    if is_gas:
        # Apply conversion only to rows where the unit was Gigajoule (the old data)
        mask_old = combined['Year'] < 2007
        combined.loc[mask_old, 'OBS_VALUE'] = combined.loc[mask_old, 'OBS_VALUE'] / 277.78
    
    # Filter timeframe
    combined = combined[(combined['Year'] >= START_YEAR) & (combined['Year'] <= END_YEAR)]
    
    # Create clean Metric names for pivoting
    combined['Country'] = np.where(combined['geo'].str.contains('Germany'), 'Germany', 'EU')
    combined['Tax_Status'] = np.where(combined['tax'].str.contains('Excluding'), 'WithoutTaxes', 'WithTaxes')
    
    # Group by Year, Country, Tax_Status and average the semesters (S1, S2)
    final = combined.groupby(['Year', 'Country', 'Tax_Status'])['OBS_VALUE'].mean().unstack(level=[1, 2])
    
    # Flatten column names: e.g., "Price_Germany_WithTaxes_perKWh"
    prefix = "Gas" if is_gas else "Elec"
    final.columns = [f"{prefix}_{country}_{tax}_perKWh" for country, tax in final.columns]
    
    return final.reset_index()

# --- 2. Run Processing ---

# Energy Mix
df_mix = datasets["energy_mix"].copy()
df_mix['Year'] = get_clean_year(df_mix)
energy_cols = ['Mineral Oils', 'Gases', 'Electricity', 'District Heating', 
               'Coal', 'Renewable Energies', 'Other Energy Sources', 'All Energy Sources']
for col in energy_cols:
    if col in df_mix.columns:
        df_mix[col] = df_mix[col].astype(float) * 1_000_000_000 # TWh to KWh
df_energy_final = df_mix[(df_mix['Year'] >= START_YEAR) & (df_mix['Year'] <= END_YEAR)].reset_index(drop=True)

# Prices
df_elec_final = process_prices(datasets["elec_prices_old"], datasets["elec_prices_new"], is_gas=False)
df_gas_final = process_prices(datasets["gas_prices_old"], datasets["gas_prices_new"], is_gas=True)

# Income
df_inc = datasets["income"].copy()
df_inc['Year'] = get_clean_year(df_inc)
df_inc = df_inc[(df_inc['Year'] >= START_YEAR) & (df_inc['Year'] <= END_YEAR)]

# Clean income geography names
df_inc['geo_short'] = np.where(df_inc['geo'] == 'Germany', 'Germany', 'EU')

# Pivot Deciles
df_income_final = df_inc.pivot_table(
    index='Year', 
    columns=['geo_short', 'quantile'], 
    values='OBS_VALUE',
    aggfunc='mean' # In case of duplicate reporting
)
df_income_final.columns = [f"Income_{geo}_{q.replace(' ', '_')}" for geo, q in df_income_final.columns]
df_income_final = df_income_final.reset_index()